In [1]:
import pandas as pd
import numpy as np
import warnings
from src.config import *

warnings.filterwarnings('ignore')
pd.options.mode.chained_assignment = None

In [2]:
# ASOS 컬럼 한글 → 영문 매핑
ASOS_COL_MAP = {
    '평균기온(°C)':       'avg_temp',
    '최저기온(°C)':       'min_temp',
    '최고기온(°C)':       'max_temp',
    '일강수량(mm)':       'd_prcp',
    '평균 이슬점온도(°C)': 'avg_dew_point_temp',
    '최소 상대습도(%)':    'min_relative_humidity',
    '평균 상대습도(%)':    'avg_relative_humidity',
    '합계 일사량(MJ/m2)':  'Total_solar_radiation',
    '평균 지면온도(°C)':   'avg_ground_temp',
    '최저 초상온도(°C)':   'min_grass_temp',
}

# 생육 시기별 윈도우 정의
# (d_end: 출하일 기준 역산 마지막 날, n_days: 이동평균 일수)
# 파종기: date-118 ~ date-105 (14일)
# 정식기: date-104 ~ date-43  (62일)
# 생육기: date-42  ~ date-21  (22일)
GROWTH_WINDOWS = {
    'Sowing':   (105, 14),
    'Planting': (43,  62),
    'Growing':  (21,  22),
}

# 각 생육 구간의 평균 값을 구하기 위한 feature 변수
FEATURE_VARS = list(ASOS_COL_MAP.values()) + ['SPI1', 'SPI2', 'SPI3', 'SPI4']

REGION_LIST  = ['강릉', '대관령']
TRAIN_START  = 2007
TRAIN_END    = 2018
ALL_END      = 2025

In [3]:
## price 가격에 대한 결측치를 forward fill로 채우는 과정을 굳이 여기서 해야하나? price 전처리때 할 수 있는거 아닌가? -> load_price로 이관할 예정
# 가격 데이터 로드 (지역 공통)
price_raw = pd.read_csv(PROCESSED_DIR + 'price_배추_가락시장_2002_2025.csv')
price_raw['DATE'] = pd.to_datetime(price_raw['DATE'])
price_filled = price_raw.set_index('DATE')['평균가격'] # DATE를 인덱스로 올리고, 평균가격 컬럼 데이터만 조회한다.

print(f'가격 데이터: {price_filled.first_valid_index().date()} ~ {price_filled.last_valid_index().date()}')
print(f'NaN 잔존: {price_filled.isnull().sum()}개')

가격 데이터: 2002-01-03 ~ 2025-12-31
NaN 잔존: 2개


In [4]:
def build_dataset(region):
    # ASOS 로드 및 컬럼 정리
    asos = pd.read_csv(PROCESSED_DIR + f'asos_{region}_2002_2025.csv')
    asos['일시'] = pd.to_datetime(asos['일시']) # 생육시기 계산을 위해서 datetime으로 변환하는거임 join과 문제 ㄴㄴ
    asos = asos.rename(columns=ASOS_COL_MAP).set_index('일시')

    # SPI 로드
    spi = pd.read_csv(PROCESSED_DIR + f'spi_{region}_2002_2025.csv')
    spi['일시'] = pd.to_datetime(spi['일시']) # 생육시기 계산을 위해서 datetime으로 변환하는거임 join과 문제 ㄴㄴ
    spi = spi.set_index('일시')[['SPI1', 'SPI2', 'SPI3', 'SPI4']]

    # ASOS + SPI 병합 (날짜 기준 outer join)
    daily = asos.join(spi, how='outer')

    # outer join 직후 NaN 확인
    print("ASOS DF row 개수:", len(asos))
    print("SPI DF row 개수:", len(spi))
    print("ASOS 날짜범위:", asos.index.min(), "~", asos.index.max())
    print("SPI  날짜범위:", spi.index.min(), "~", spi.index.max())
    nan_check = daily.isnull().sum()
    print("outer join 후 NaN:\n", nan_check[nan_check > 0])

    rows = [] # 이 변수가 최종 산출물 구조(캡처본)을 저장하기 위한 변수
    for year in range(TRAIN_START, ALL_END + 1):
        for date in pd.date_range(f'{year}-07-01', f'{year}-10-31'):

            # 가격 조회 (현재일, 전년 동일일)
            price_today    = price_filled.get(date)
            price_lastyear = price_filled.get(date - pd.DateOffset(years=1))

            # 조건에 하나라도 true가 걸리면 다음 날짜로 넘어감
            if pd.isna(price_today) or pd.isna(price_lastyear) or price_lastyear == 0:
                continue

            # 피처 계산
            # 생육 구간별로 피처를 계산
            row = {'date': date}
            for period, (d_end, n_days) in GROWTH_WINDOWS.items():
                end   = date - pd.Timedelta(days=d_end) # 날짜별로 생육기별 구간의 끝날짜
                start = end  - pd.Timedelta(days=n_days - 1) # 날짜별로 생육기별 구간의 시작일자
                window = daily.loc[start:end]

                for var in FEATURE_VARS:
                    if var in window.columns:
                        row[f'{period}_{var}'] = window[var].mean() # 평균값을 구함
                    else:
                        row[f'{period}_{var}'] = np.nan

            # 타겟: 전년 동일일 대비 가격 변동률
            row['price_change_rate'] = (price_today - price_lastyear) / price_lastyear
            rows.append(row)

    return pd.DataFrame(rows)

In [5]:
for region in REGION_LIST:
    print(f'[{region}] 피처 생성 중...')
    df = build_dataset(region)

    # 분위수는 학습 데이터(2007~2018)에서만 산출 → 데이터 누수 방지
    train_mask  = df['date'].dt.year.between(TRAIN_START, TRAIN_END)
    train_rates = df.loc[train_mask, 'price_change_rate']
    q1, q2, q3  = train_rates.quantile([0.25, 0.50, 0.75])

    print(f'  분위수 기준 (학습 데이터) — Q1: {q1:.3f}, Q2: {q2:.3f}, Q3: {q3:.3f}')

    df['target'] = pd.cut(
        df['price_change_rate'],
        bins=[-np.inf, q1, q2, q3, np.inf],
        labels=[0, 1, 2, 3]
    ).astype(int)

    out_path = PROCESSED_DIR + f'dataset_{region}.csv'
    df.to_csv(out_path, encoding='utf-8-sig', index=False)

    print(f'  저장: {out_path}')
    print(f'  shape: {df.shape}')
    print(f'  날짜범위: {df["date"].min().date()} ~ {df["date"].max().date()}')
    print(f'  타겟 분포:\n{df["target"].value_counts().sort_index().to_string()}')
    print()

[강릉] 피처 생성 중...
ASOS DF row 개수: 8766
SPI DF row 개수: 8766
ASOS 날짜범위: 2002-01-01 00:00:00 ~ 2025-12-31 00:00:00
SPI  날짜범위: 2002-01-01 00:00:00 ~ 2025-12-31 00:00:00
outer join 후 NaN:
 Series([], dtype: int64)
  분위수 기준 (학습 데이터) — Q1: -0.305, Q2: 0.015, Q3: 0.537
  저장: /Users/jeongseok/Desktop/업무/2. 실측가뭄/4. 가뭄영향정보 생산 기술 개발/6. 실측가뭄_농산물 도매가격 예측모델_스크립트/code//processed/dataset_강릉.csv
  shape: (2337, 45)
  날짜범위: 2007-07-01 ~ 2025-10-31
  타겟 분포:
target
0    625
1    521
2    558
3    633

[대관령] 피처 생성 중...
ASOS DF row 개수: 8766
SPI DF row 개수: 8766
ASOS 날짜범위: 2002-01-01 00:00:00 ~ 2025-12-31 00:00:00
SPI  날짜범위: 2002-01-01 00:00:00 ~ 2025-12-31 00:00:00
outer join 후 NaN:
 Series([], dtype: int64)
  분위수 기준 (학습 데이터) — Q1: -0.305, Q2: 0.015, Q3: 0.537
  저장: /Users/jeongseok/Desktop/업무/2. 실측가뭄/4. 가뭄영향정보 생산 기술 개발/6. 실측가뭄_농산물 도매가격 예측모델_스크립트/code//processed/dataset_대관령.csv
  shape: (2337, 4